In [ ]:
import ROOT

ROOT_FILE = "/data2/segmentlinking/CMSSW_12_5_0_pre3/RelValTTbar_14TeV_CMSSW_12_5_0_pre3/event_1000.root"
TREE_PATH = "trackingNtuple/tree"

OUTPUT_FILE = "tree_structure.txt"


In [ ]:
# ─────────────────────────────────────────────
# open file and tree
# ─────────────────────────────────────────────

f = ROOT.TFile.Open(ROOT_FILE)
t = f.Get(TREE_PATH)

print("Opened tree with", t.GetEntries(), "events")


# ─────────────────────────────────────────────
# save branch structure to file
# ─────────────────────────────────────────────

ROOT.gSystem.RedirectOutput(OUTPUT_FILE, "w")
t.Print()
ROOT.gSystem.RedirectOutput("", "")

print(f"Tree structure written to {OUTPUT_FILE}")


# ─────────────────────────────────────────────
# inspect first few events
# ─────────────────────────────────────────────

N_EVENTS = 5

print("\nInspecting first events:\n")

for i in range(min(N_EVENTS, t.GetEntries())):

    t.GetEntry(i)

    # example sim-track branches
    # adjust names after we inspect tree_structure.txt

    if hasattr(t, "sim_pt"):
        n = len(t.sim_pt)
        print(f"Event {i}: sim tracks = {n}")

        if n > 0:
            print("   first sim_pt:", t.sim_pt[0])

    if hasattr(t, "sim_eta"):
        print("   sim_eta size:", len(t.sim_eta))

    if hasattr(t, "sim_phi"):
        print("   sim_phi size:", len(t.sim_phi))

In [ ]:
#imports for the root file rip

from __future__ import annotations

import math
import os
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import ROOT

In [ ]:
# Paralell data rip, splits up into n-chunks then writes each to a npz file then we merge

# we implement some filters like the naive eta limitation etc

# ─────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────

ROOT_FILE = "/data2/segmentlinking/CMSSW_12_5_0_pre3/RelValTTbar_14TeV_CMSSW_12_5_0_pre3/event_1000.root"
TREE_PATH = "trackingNtuple/tree"

OUTDIR = Path("track_chunks_etaM1to1")
FINAL_OUTFILE = "track_data_etaM1to1.npz"

ETA_MIN = -1.0
ETA_MAX = 1.0
REQUIRE_NVALID_POSITIVE = True

N_WORKERS = 20

FEATURE_BRANCHES = [
    "sim_pt",
    "sim_eta",
    "sim_phi",
    "sim_q",
    "sim_nValid",
    "sim_nPixel",
    "sim_nStrip",
    "sim_nLay",
]

LABEL_BRANCHES = [
    "sim_pca_pt",
    "sim_pca_eta",
    "sim_pca_phi",
    "sim_pca_dxy",
    "sim_pca_dz",
]


# ─────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────

def vec_to_np(v, dtype=np.float32):
    return np.asarray(list(v), dtype=dtype)


def all_same_length(arrays: list[np.ndarray]) -> bool:
    if not arrays:
        return True
    n = len(arrays[0])
    return all(len(a) == n for a in arrays)


def split_ranges(n_events: int, n_chunks: int) -> list[tuple[int, int, int]]:
    """
    Returns list of (chunk_id, start_evt, end_evt_exclusive)
    """
    ranges = []
    chunk_size = math.ceil(n_events / n_chunks)
    for chunk_id in range(n_chunks):
        start = chunk_id * chunk_size
        end = min((chunk_id + 1) * chunk_size, n_events)
        if start < end:
            ranges.append((chunk_id, start, end))
    return ranges


# ─────────────────────────────────────────────
# Worker
# ─────────────────────────────────────────────

def process_chunk(chunk_id: int, evt_start: int, evt_end: int, outdir_str: str):
    outdir = Path(outdir_str)
    outdir.mkdir(parents=True, exist_ok=True)

    f = ROOT.TFile.Open(ROOT_FILE)
    if not f or f.IsZombie():
        raise RuntimeError(f"[chunk {chunk_id}] Could not open ROOT file")

    t = f.Get(TREE_PATH)
    if not t:
        raise RuntimeError(f"[chunk {chunk_id}] Could not get tree {TREE_PATH}")

    feature_chunks = []
    label_chunks = []

    n_total_rows = 0
    n_kept_rows = 0
    n_bad_length_events = 0

    for i_evt in range(evt_start, evt_end):
        t.GetEntry(i_evt)

        feat_arrays = [
            vec_to_np(getattr(t, name), dtype=np.float32)
            for name in FEATURE_BRANCHES
        ]
        lab_arrays = [
            vec_to_np(getattr(t, name), dtype=np.float32)
            for name in LABEL_BRANCHES
        ]

        all_arrays = feat_arrays + lab_arrays
        if not all_same_length(all_arrays):
            n_bad_length_events += 1
            continue

        n = len(feat_arrays[0])
        if n == 0:
            continue

        n_total_rows += n

        X_evt = np.column_stack(feat_arrays).astype(np.float32, copy=False)
        Y_evt = np.column_stack(lab_arrays).astype(np.float32, copy=False)

        eta = X_evt[:, 1]
        mask = (eta >= ETA_MIN) & (eta <= ETA_MAX)

        mask &= np.all(np.isfinite(X_evt), axis=1)
        mask &= np.all(np.isfinite(Y_evt), axis=1)

        if REQUIRE_NVALID_POSITIVE:
            mask &= (X_evt[:, 4] > 0)

        X_evt = X_evt[mask]
        Y_evt = Y_evt[mask]

        if len(X_evt) > 0:
            feature_chunks.append(X_evt)
            label_chunks.append(Y_evt)
            n_kept_rows += len(X_evt)

    if feature_chunks:
        X = np.concatenate(feature_chunks, axis=0)
        Y = np.concatenate(label_chunks, axis=0)
    else:
        X = np.empty((0, len(FEATURE_BRANCHES)), dtype=np.float32)
        Y = np.empty((0, len(LABEL_BRANCHES)), dtype=np.float32)

    outfile = outdir / f"chunk_{chunk_id:03d}.npz"
    np.savez_compressed(
        outfile,
        X=X,
        Y=Y,
        evt_start=evt_start,
        evt_end=evt_end,
        n_total_rows=n_total_rows,
        n_kept_rows=n_kept_rows,
        n_bad_length_events=n_bad_length_events,
    )

    return {
        "chunk_id": chunk_id,
        "evt_start": evt_start,
        "evt_end": evt_end,
        "outfile": str(outfile),
        "shape_X": X.shape,
        "shape_Y": Y.shape,
        "n_total_rows": n_total_rows,
        "n_kept_rows": n_kept_rows,
        "n_bad_length_events": n_bad_length_events,
    }


# ─────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────

def main():
    OUTDIR.mkdir(parents=True, exist_ok=True)

    f = ROOT.TFile.Open(ROOT_FILE)
    if not f or f.IsZombie():
        raise RuntimeError(f"Could not open ROOT file: {ROOT_FILE}")

    t = f.Get(TREE_PATH)
    if not t:
        raise RuntimeError(f"Could not get tree: {TREE_PATH}")

    n_events = t.GetEntries()
    print(f"Opened tree with {n_events} events")

    ranges = split_ranges(n_events, N_WORKERS)
    print(f"Using {len(ranges)} worker chunks")

    results = []
    with ProcessPoolExecutor(max_workers=len(ranges)) as ex:
        futures = [
            ex.submit(process_chunk, chunk_id, start, end, str(OUTDIR))
            for chunk_id, start, end in ranges
        ]

        for fut in as_completed(futures):
            res = fut.result()
            results.append(res)
            print(
                f"[done] chunk {res['chunk_id']:03d} "
                f"events {res['evt_start']}:{res['evt_end']} "
                f"X{res['shape_X']} Y{res['shape_Y']} "
                f"kept {res['n_kept_rows']:,}/{res['n_total_rows']:,}"
            )

    results.sort(key=lambda r: r["chunk_id"])

    X_parts = []
    Y_parts = []

    total_rows = 0
    kept_rows = 0
    bad_events = 0

    for res in results:
        data = np.load(res["outfile"], allow_pickle=True)
        X_parts.append(data["X"])
        Y_parts.append(data["Y"])
        total_rows += int(data["n_total_rows"])
        kept_rows += int(data["n_kept_rows"])
        bad_events += int(data["n_bad_length_events"])

    X = np.concatenate(X_parts, axis=0) if X_parts else np.empty((0, len(FEATURE_BRANCHES)), dtype=np.float32)
    Y = np.concatenate(Y_parts, axis=0) if Y_parts else np.empty((0, len(LABEL_BRANCHES)), dtype=np.float32)

    np.savez_compressed(
        FINAL_OUTFILE,
        X=X,
        Y=Y,
        feature_names=np.array(FEATURE_BRANCHES, dtype=object),
        label_names=np.array(LABEL_BRANCHES, dtype=object),
    )

    print("\nFinal merged dataset:")
    print("X shape:", X.shape)
    print("Y shape:", Y.shape)
    print(f"Total raw rows: {total_rows:,}")
    print(f"Total kept rows: {kept_rows:,}")
    print(f"Bad-length events skipped: {bad_events}")
    print(f"Saved final dataset to {FINAL_OUTFILE}")


if __name__ == "__main__":
    main()